# 04: Regression Model (Learning Outcome 3)

**Learning Outcome 3 (LO3):** Regression modeling and predictive analytics

**Model:** `yield_loss ~ monitoring_index + climate_risk + temp + precip`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Prepare regression data
X = valid_data[['monitoring_index', 'climate_risk_index']].values
y = valid_data['yield_loss_pct'].values

# Add constant
X = sm.add_constant(X)

# Fit OLS regression
model = sm.OLS(y, X).fit()

# Print summary
print("\n" + "="*70)
print("REGRESSION RESULTS: Yield Loss Model")
print("="*70)
print(model.summary())

# Model diagnostics
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Actual vs Predicted
predictions = model.predict(X)
axes[0, 0].scatter(y, predictions, alpha=0.6, s=100, edgecolors='black')
axes[0, 0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Yield Loss (%)')
axes[0, 0].set_ylabel('Predicted Yield Loss (%)')
axes[0, 0].set_title('Actual vs Predicted', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# 2. Residuals vs Fitted
residuals = model.resid
axes[0, 1].scatter(predictions, residuals, alpha=0.6, s=100, edgecolors='black')
axes[0, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0, 1].set_xlabel('Fitted Values')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Residual Plot', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# 3. Q-Q Plot
stats.probplot(residuals, dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot', fontweight='bold')

# 4. Histogram of Residuals
axes[1, 1].hist(residuals, bins=8, edgecolor='black', alpha=0.7, color='skyblue')
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Residual Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('results/graphs/04_regression_diagnostics.png', dpi=300, bbox_inches='tight')
plt.show()

# Model performance metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(y, predictions)
rmse = np.sqrt(mean_squared_error(y, predictions))
mape = np.mean(np.abs((y - predictions) / (y + 1e-10))) * 100

print("\n=== Model Performance Metrics ===")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")

# Summary dataframe
metrics_df = pd.DataFrame({
    'Metric': ['R-squared', 'Adj. R-squared', 'AIC', 'BIC', 'MAE', 'RMSE', 'MAPE (%)'],
    'Value': [f'{model.rsquared:.4f}', f'{model.rsquared_adj:.4f}', 
             f'{model.aic:.2f}', f'{model.bic:.2f}', f'{mae:.4f}', 
             f'{rmse:.4f}', f'{mape:.2f}']
})
print("\n")
print(metrics_df.to_string(index=False))